# Part 1 — Three Kinds of Problems, With Numbers Attached

Same approach as the desire-paths notebook: small, clearly-named functions instead of one long block, and instead of just eyeballing a plot and saying "that looks more/less predictable," we'll actually **measure** how predictable each case is, with a simple test repeated throughout:

> Change one small thing (repeat the run exactly, use a different random seed, or nudge the starting conditions slightly) — how much does the result move?

We'll ask that question of all three of Weaver's cases and put real numbers next to the answer, so "Case 3 is harder" isn't just an assertion by the end of this notebook.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.integrate import odeint

plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True

print("Ready.")


---
## Case 1 — "Simple" problems

**Everyday example:** you throw a ball. Two numbers — launch angle and speed — determine the entire flight path. No individuals, no randomness, nothing hidden.


In [ ]:
GRAVITY = 9.81
LAUNCH_SPEED = 20.0

def ball_path(angle_degrees):
    """Return the (x, y) flight path of a ball thrown at `angle_degrees`.

    Exactly the same angle in always produces exactly the same path out --
    there is nothing here that could vary between two runs.
    """
    angle = np.radians(angle_degrees)
    vx, vy = LAUNCH_SPEED * np.cos(angle), LAUNCH_SPEED * np.sin(angle)
    flight_time = 2 * vy / GRAVITY
    t = np.linspace(0, flight_time, 200)
    x = vx * t
    y = vy * t - 0.5 * GRAVITY * t**2
    return x, y

def landing_distance(angle_degrees):
    """Just how far the ball travels before it lands."""
    x, _ = ball_path(angle_degrees)
    return x[-1]


In [ ]:
plt.figure()
for angle in [20, 35, 45, 60, 75]:
    x, y = ball_path(angle)
    plt.plot(x, y, label=f"{angle}°")
plt.title("A ball thrown at different angles")
plt.xlabel("distance traveled")
plt.ylabel("height")
plt.legend(title="launch angle")
plt.ylim(bottom=0)
plt.show()


### Summary stat: how much does the result change if we repeat it?

This is the baseline for everything that follows in this notebook. Run the exact same throw twice and compare.


In [ ]:
run_1 = landing_distance(45)
run_2 = landing_distance(45)

case1_stats = pd.DataFrame([{
    "case": "1. Simplicity",
    "what we changed": "nothing -- exact repeat",
    "what we measured": "landing distance",
    "run 1": round(run_1, 6),
    "run 2": round(run_2, 6),
    "difference": round(abs(run_1 - run_2), 6),
}]).set_index("case")

case1_stats


**Takeaway:** the difference is exactly 0. Not "very small" -- exactly zero. That's the benchmark every other case in this notebook gets compared against.


---
## Case 2 — "Disorganized" problems

**Everyday example:** hundreds of people wandering a plaza with no destination, bumping and changing direction at random -- or gas molecules bouncing around a box. No individual path is predictable. Let's check whether the *group* is.


In [ ]:
def create_particles(n_particles, box_size, rng):
    """Random starting position and velocity for each particle."""
    positions = rng.uniform(0, box_size, size=(n_particles, 2))
    velocities = rng.normal(0, 1.0, size=(n_particles, 2))
    return positions, velocities

def step_particles(positions, velocities, box_size, dt):
    """Move every particle one small time step, bouncing off the walls."""
    positions += velocities * dt
    for dim in range(2):
        hit_wall = (positions[:, dim] < 0) | (positions[:, dim] > box_size)
        velocities[hit_wall, dim] *= -1
        positions[:, dim] = np.clip(positions[:, dim], 0, box_size)
    return positions, velocities

def run_particle_simulation(seed, n_particles=800, box_size=10.0, n_steps=300, dt=0.05):
    """Run the whole simulation and return final positions and speeds."""
    rng = np.random.default_rng(seed)
    positions, velocities = create_particles(n_particles, box_size, rng)
    for _ in range(n_steps):
        positions, velocities = step_particles(positions, velocities, box_size, dt)
    speeds = np.linalg.norm(velocities, axis=1)
    return positions, speeds


In [ ]:
positions, speeds = run_particle_simulation(seed=7)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(positions[:20, 0], positions[:20, 1], alpha=0.7)
axes[0].set_title("Where 20 sample particles ended up")
axes[0].set_xlim(0, 10); axes[0].set_ylim(0, 10)

axes[1].hist(speeds, bins=30, color="steelblue", edgecolor="white")
axes[1].set_title(f"Speed of all {len(speeds)} particles")
plt.tight_layout()
plt.show()


### Summary stat: individuals vs. the group, across 5 different random seeds

We'll track two things for each seed: where one specific particle (#0) ends up, and what the *average* speed of the whole population is.


In [ ]:
records = []
for seed in [1, 2, 3, 4, 5]:
    positions, speeds = run_particle_simulation(seed=seed)
    records.append({
        "seed": seed,
        "particle #0's final x-position": round(positions[0, 0], 3),
        "average speed, all particles": round(speeds.mean(), 4),
    })

case2_runs = pd.DataFrame(records)
case2_runs


In [ ]:
box_width = 10.0
position_spread_pct = 100 * (case2_runs["particle #0's final x-position"].max()
                              - case2_runs["particle #0's final x-position"].min()) / box_width
speed_spread_pct = 100 * (case2_runs["average speed, all particles"].std()
                           / case2_runs["average speed, all particles"].mean())

print(f"One particle's position swings across {position_spread_pct:.0f}% of the box, run to run.")
print(f"The population's average speed varies by only {speed_spread_pct:.1f}% of its mean, run to run.")


**Takeaway:** you cannot predict where particle #0 will be -- it's all over the box depending on the random seed. But the *average speed of the whole population* barely moves at all between runs. Individually unpredictable, collectively stable -- exactly Weaver's disorganized-complexity claim, now with a number attached to each half of it.


---
## Case 3 — "Organized" problems, the classical (equation-based) attempt

**Everyday example:** an ecosystem -- grass, rabbits, foxes, and parasites, all affecting each other. The historical approach: write one equation per population, describing the *group total*, with no individuals at all.


In [ ]:
def ecosystem(populations, t):
    """How fast each population is changing right now, given all four
    current population sizes. No individual animals here -- just totals.
    """
    grass, rabbits, foxes, parasites = populations
    d_grass     = 1.0 * grass * (1 - grass / 50) - 0.04 * grass * rabbits
    d_rabbits   = 0.02 * grass * rabbits - 0.05 * rabbits * foxes - 0.01 * rabbits * parasites - 0.1 * rabbits
    d_foxes     = 0.02 * rabbits * foxes - 0.03 * foxes * parasites - 0.2 * foxes
    d_parasites = 0.015 * foxes * parasites + 0.005 * rabbits * parasites - 0.15 * parasites
    return [d_grass, d_rabbits, d_foxes, d_parasites]

def run_ecosystem(starting_populations, t_max=200, n_points=2000):
    """Solve the equations forward in time from a given starting point."""
    t = np.linspace(0, t_max, n_points)
    solution = odeint(ecosystem, starting_populations, t)
    return t, solution


In [ ]:
starting_populations = [40, 9, 5, 2]   # grass, rabbits, foxes, parasites
t, solution = run_ecosystem(starting_populations)

plt.figure()
for i, name in enumerate(["Grass", "Rabbits", "Foxes", "Parasites"]):
    plt.plot(t, solution[:, i], label=name)
plt.title("Four interdependent populations over time")
plt.xlabel("time")
plt.ylabel("population size")
plt.legend()
plt.show()


### Summary stat: what happens if we nudge the starting populations very slightly?

Case 1 was exactly reproducible by construction -- there's nothing random in it at all. Case 3 is *also* built with no randomness anywhere. So let's ask the more interesting version of the reproducibility question: if we start from *slightly different* populations (a stand-in for "history could have gone a little differently"), does the long-run outcome actually change?


In [ ]:
rng = np.random.default_rng(0)
base = np.array(starting_populations)

records = []
for trial in range(8):
    nudged_start = base * (1 + rng.normal(0, 0.001, size=4))   # 0.1% random nudge
    _, sol = run_ecosystem(nudged_start)
    final = sol[-1]
    records.append({
        "trial": trial + 1,
        "grass (final)": round(final[0], 4),
        "rabbits (final)": round(final[1], 4),
        "foxes (final)": round(final[2], 4),
        "parasites (final)": round(final[3], 4),
    })

case3_runs = pd.DataFrame(records)
case3_runs


In [ ]:
spreads = case3_runs[["grass (final)", "rabbits (final)", "foxes (final)", "parasites (final)"]].agg(
    lambda col: 100 * (col.max() - col.min()) / col.mean()
)
print("Spread across 8 trials, as % of the mean final value:")
print(spreads.round(6))


**Takeaway, and this is the important one:** even a 0.1% nudge to where we started produces essentially **0% difference** in where the system ends up. The equations pull every nearby starting point toward the exact same long-run equilibrium.

That's a strange result to sit with. We built this model specifically to handle "organized complexity" -- Weaver's whole point was that these problems resist being reduced to simple, guaranteed-predictable outcomes. But this equation-based version turned out to be **just as perfectly predictable as Case 1's ball throw**, just with a longer formula. History (which starting point you nudge to) barely matters here at all.

That's not a coincidence, and it's not really a success. It's a symptom of what we already noticed in Notebook 1's discussion: this model works precisely because it throws away everything that could make history matter -- there are no individuals, no locations, no encounters, no luck. Whatever "organized complexity" really requires, an equation with a stable attractor apparently isn't forced to provide it.


## Putting all three side by side

Same test, applied consistently to all three cases: change one small thing, measure how much the outcome moves.


In [ ]:
comparison = pd.DataFrame([
    {
        "case": "1. Simplicity",
        "what we varied": "nothing (exact repeat)",
        "what we measured": "landing distance",
        "spread across trials": "0.0% (exact)",
    },
    {
        "case": "2. Disorganized -- one individual",
        "what we varied": "random seed",
        "what we measured": "particle #0's position",
        "spread across trials": f"~{position_spread_pct:.0f}% of the box width",
    },
    {
        "case": "2. Disorganized -- the group",
        "what we varied": "random seed",
        "what we measured": "average speed, all particles",
        "spread across trials": f"~{speed_spread_pct:.1f}% of the mean",
    },
    {
        "case": "3. Organized (equations)",
        "what we varied": "starting populations, ±0.1%",
        "what we measured": "final population sizes",
        "spread across trials": "~0.0% (converges to the same equilibrium)",
    },
]).set_index("case")

comparison


**What should jump out:** Case 1 and Case 3 ended up in the same place -- both essentially 0% spread, both fully predictable once you know the setup. The only genuinely different behavior in this whole notebook is Case 2's *individual* particle, and that unpredictability doesn't survive to the group level either.

Nowhere in this notebook does the aggregate outcome itself refuse to be predicted. That's the gap Part 2 exists to fill -- not with a longer equation, but by giving up equations for the whole group entirely, and watching what happens when individuals interact through simple local rules instead.


## Recap

| | Case 1: Simple | Case 2: Disorganized | Case 3: Organized (equations) |
|---|---|---|---|
| how many things involved | 2–3 | thousands, independent | a handful, interdependent |
| individually predictable? | yes, exactly | no | n/a -- no individuals modeled |
| group-level outcome reproducible? | trivially, yes | yes, tightly | yes, surprisingly tightly |
| spread across repeated/perturbed trials | 0% | ~3% (aggregate) | ~0% |

**Next up (Part 2):** an agent-based model where that last row finally stops being ~0%.
